In [1]:
import pathlib
import sys

_here = pathlib.Path.cwd().resolve()
for _parent in [_here, *_here.parents]:
    if (_parent / "src" / "quant_textbook").exists():
        sys.path.insert(0, str(_parent / "src"))
        break

# 48. B9 — Deep learning and SEC filing representations

> B9の目的はneural architectureを使うことではない。同じ情報集合・split・計算予算で、線形probeを超える表現価値があるかを反証可能に調べることである。

## 学習目標

- computational graphからreverse-mode backpropagationを導出できる
- MLP、LSTM、TCN、self-attentionのinductive biasを比較できる
- vocabulary、imputation、scalingをactive training partitionだけでfitできる
- TF–IDF＋ridgeをdeep modelより前に固定できる
- data scale、duplicate、timestamp、budgetの差をarchitecture効果と混同しない

## 前提知識

- B4のgradient auditとoptimization trace
- B5–B6のregularization、validation、shift診断
- M6のSEC point-in-time panelとB9 pre-analysis contract

In [2]:
import time

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio

import quant_textbook as qt

pio.renderers.default = "notebook_connected"
RANDOM_SEED = 20260810
NOTEBOOK_ID = 48


def task_rng(task_id, *coordinates):
    entropy = [
        RANDOM_SEED,
        NOTEBOOK_ID,
        int(task_id),
        *(int(coordinate) for coordinate in coordinates),
    ]
    return np.random.default_rng(np.random.SeedSequence(entropy))

In [3]:
fixture = qt.load_sec_teaching_fixture()
train_mask = fixture.training_mask
validation_mask = fixture.validation_mask

assert train_mask.sum() == 192
assert validation_mask.sum() == 64
assert not np.any(fixture.target_available_dates >= np.datetime64("2023-10-23"))
assert set(fixture.partitions) == {"inner_train", "inner_validation"}

print("fixture rows:", fixture.targets.size)
print("inner train / validation:", int(train_mask.sum()), int(validation_mask.sum()))
print("numeric / sequence shape:", fixture.numeric_features.shape, fixture.token_hashes.shape)
print("locked outer rows present: False")
print("fixture hash lineage:", fixture.provenance)

fixture rows: 256
inner train / validation: 192 64
numeric / sequence shape: (256, 12) (256, 128)
locked outer rows present: False
fixture hash lineage: {'panel_artifact_sha256': '6c6008c2f28c30299e15e37613cfb0b3b22e8fd283858f5b459227c7e4a412a8', 'previous_filing_sidecar_sha256': '9ff2efef335357ff53bb1e4ba5c57f4b2e8799fc4ee5d830c55843a50026fbbc', 'normalized_manifest_sha256': '1283b9cb0992cfd2caaa942f6c869e212762c90a9abbc9a050173f5e3963daba', 'preanalysis_contract_sha256': '0aa180acbcd2b685509d6ec65fdf40f9edfcfc544ecec62c930facd0d4615b20'}


## 1. Evidence chainと範囲

| Week | Core implementation | 実データlab | 主な反証 |
|---|---|---|---|
| 33 | MLP forward/backprop/Adam | numeric feature MLP | gradient不一致、linear優位 |
| 34 | LSTM/causal TCN forward | 同じhashed token sequence | unequal budget、短いeffective context |
| 35 | scaled dot-product attention | small attention probe | mask漏れ、data不足 |
| 36 | train-only TF–IDF/ridge、ablation | previous SEC filing | target text、duplicate、coverage |
| Project | frozen contract audit | SEC Assets log-change | `no_model_selected`を許す |

教材fixtureは実SEC filingとfundamentalsから作ったが、inner train 192行・inner validation 64行だけの縮約版である。raw text、CIK、accession、locked outer 413行を含まない。したがってarchitectureの仕組みと漏洩監査には使えるが、pre-registered tournamentのnominee選定には使わない。

In [4]:
target_frame = pd.DataFrame(
    {
        "partition": fixture.partitions,
        "target": fixture.targets,
        "date": fixture.target_available_dates,
    }
)
summary = target_frame.groupby("partition")["target"].agg(["count", "mean", "std", "median"])
display(summary)

fig = go.Figure()
for partition in ["inner_train", "inner_validation"]:
    values = target_frame.loc[target_frame["partition"] == partition, "target"]
    fig.add_histogram(x=values, name=partition, opacity=0.6, histnorm="probability density")
fig.update_layout(
    title="Real SEC-derived teaching fixture: target shift",
    xaxis_title="Next-quarter log Assets change",
    barmode="overlay",
    template="plotly_white",
)
fig.show()

,count,mean,std,median
partition,,,,
inner_train,192,0.020072,0.093454,0.007943
inner_validation,64,-0.013754,0.114545,0.004372


## 2. Claim boundary

estimandはfixed-anchor feasibility cohortにおける

$$
E\!\left[\log(A_{i,t}/A_{i,t-1})\mid\mathcal I_{i,t-1}\right]
$$

の予測である。filing languageの因果効果、abnormal return、取引収益、全米issuerへの代表性は主張しない。fixtureは本文でなくmany-to-one token bucketだけを保持するが、dictionary attackに対するprivacy保証ではない。正式candidateの512-token chunk契約も置き換えない。

## 3. 失敗モード

- target accessionの文書をfeatureへ入れる
- outer結果を見てarchitecture、seed、thresholdを変更する
- vocabularyやstandardizationをvalidation込みでfitする
- parameter数、epoch、run数が違うmodelを「同じ予算」と呼ぶ
- pretrained modelの一般知識を無料の情報とみなす
- deep modelを採用しない結論を失敗扱いする

## 4. 段階別演習

### 基礎

1. fixtureとfull pre-analysis datasetの違いを5項目書け。
2. `known_at`より後に利用可能なtarget historyを使えない理由を説明せよ。

### 標準

3. data、information set、parameter budget、metricを揃えたcomparison tableを作れ。
4. text-only / numeric-only / joint ablationが回答する問いを分けよ。

### 研究

5. pretrained encoderをAdvancedへ追加する際のdependency、license、compute、leakage監査を書け。

## 5. Exit Criteria

- [ ] 教材fixtureをcandidate tournamentと呼んでいない
- [ ] outer testを一度も読んでいない
- [ ] linear baselineをneural modelより先に置いた
- [ ] architecture効果とdata/budget差を分離した
- [ ] `no_model_selected`を有効な結論にした

## 6. 出典


- [Goodfellow, Bengio, and Courville, *Deep Learning*](https://www.deeplearningbook.org/)
- [Glorot and Bengio (2010), Understanding the difficulty of training deep feedforward neural networks](https://proceedings.mlr.press/v9/glorot10a.html)
- [Kingma and Ba (2015), Adam](https://arxiv.org/abs/1412.6980)

- [Hochreiter and Schmidhuber (1997), Long Short-Term Memory](https://www.bioinf.jku.at/publications/older/2604.pdf)
- [Bai, Kolter, and Koltun (2018), An Empirical Evaluation of Generic Convolutional and Recurrent Networks](https://arxiv.org/abs/1803.01271)
- [Vaswani et al. (2017), Attention Is All You Need](https://arxiv.org/abs/1706.03762)

- [SEC EDGAR application programming interfaces](https://www.sec.gov/search-filings/edgar-application-programming-interfaces)
- [SEC Developer Resources](https://www.sec.gov/about/developer-resources)
- [Manning, Raghavan, and Schütze, *Introduction to Information Retrieval*](https://nlp.stanford.edu/IR-book/)